In [42]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Milestone 1

1. Calculate the frequency distribution of the correct  answer  (A, B, C, D, E) in train.csv. Based on your counts, what is the sum of the occurrences of the most frequent option and the least frequent option?  

In [43]:
import pandas as pd
import string

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

freq = train["answer"].value_counts()

print(freq)

most_frequent = freq.max()
least_frequent = freq.min()

result = most_frequent + least_frequent

print("Most Frequent:", most_frequent)
print("Least Frequent:", least_frequent)
print("Sum:", result)

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
Most Frequent: 490
Least Frequent: 324
Sum: 814


2. After converting the prompt column to lowercase and removing all standard punctuation characters (using Python's string.punctuation), split the text by whitespace. What is the total number of unique words (vocabulary size) across the entire cleaned prompt column of train.csv?  

In [44]:
prompts = train["prompt"].astype(str).str.lower()

translator = str.maketrans('', '', string.punctuation)
prompts = prompts.str.translate(translator)

vocab = set()

for text in prompts:
    vocab.update(text.split())


print("Vocabulary Size:", len(vocab))

Vocabulary Size: 859


3. Using the cleaned prompt from Row ID 1, filter out the standard English stop words using sklearn.feature_extraction.text.ENGLISH_STOP_WORDS. How many words are left in the prompt for Row ID 1 after filtering?  

In [45]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

text = train.loc[train["id"] == 1, "prompt"].iloc[0]

text = text.lower()

text = text.translate(
    str.maketrans('', '', string.punctuation)
)

words = text.split()

filtered_words = [
    w for w in words
    if w not in ENGLISH_STOP_WORDS
]
print(len(filtered_words))

13


4. Fit a default TfidfVectorizer(stop_words='english') on a list containing all the combined text of the prompts and options in train.csv. What is the exact total number of feature columns (vocabulary size) generated by the vectorizer?  

In [46]:
all_texts = []
for _, row in train.iterrows():
    all_texts.append(row['prompt'])
    all_texts.append(row['A'])
    all_texts.append(row['B'])
    all_texts.append(row['C'])
    all_texts.append(row['D'])
    all_texts.append(row['E'])
 
vectorizer = TfidfVectorizer(stop_words='english')
vectorizer.fit(all_texts)
 
print(f"TF-IDF vocabulary size: {len(vectorizer.vocabulary_)}")



TF-IDF vocabulary size: 2762


5. Using the TF-IDF vectorizer fitted in Question 3, calculate the cosine similarity between the prompt and option A strictly for Row ID 1. What is the resulting similarity score? (Round to 4 decimal places).  

In [47]:
row1 = train[train['id'] == 1].iloc[0]
 
prompt_vec = vectorizer.transform([row1['prompt']])
option_vec = vectorizer.transform([row1['A']])
 
similarity = cosine_similarity(prompt_vec, option_vec)[0][0]
print(f"Cosine similarity: {round(similarity, 4)}")

Cosine similarity: 0.2328


6. Expand the logic from Question 4: For every row in train.csv, calculate the cosine similarity between the prompt and each of its 5 options .  Then calculate the percentage of instances where the option with the highest cosine similarity matches the correct answer.   

In [48]:
correct = 0
 
for _,row in train.iterrows():
    prompt_vec = vectorizer.transform([row['prompt']])
 
    sims = {}
    for opt in ['A', 'B', 'C', 'D', 'E']:
        opt_vec = vectorizer.transform([row[opt]])
        sims[opt] = cosine_similarity(prompt_vec, opt_vec)[0][0]
 
    best_option = max(sims, key=sims.get)
    if best_option == row['answer']:
        correct += 1
 
accuracy = correct / len(train) * 100
print(f"Correct: {correct} / {len(train)}")
print(f"Accuracy: {round(accuracy, 4)}%")

Correct: 274 / 2000
Accuracy: 13.7%


7. If the ground truth answer for a question is C, what is the MAP@3 score if a model predicts C A B?  

In [49]:
def ap_at_3(true_answer, predictions):
    for rank, pred in enumerate(predictions[:3], start=1):
        if pred == true_answer:
            return 1.0 / rank
    return 0.0
 
score=ap_at_3('C', ['C', 'A', 'B'])
print(f"MAP@3 score: {score}")

MAP@3 score: 1.0


8. If the ground truth answer for a question is  B, what is the MAP@3 score if a model predicts D B E?  

In [50]:
score=ap_at_3('B', ['D', 'B', 'E'])
print(f"MAP@3 score: {score}")
 

MAP@3 score: 0.5


9. The Majority Class Baseline: Find the most frequent correct answer in the training set (using your data from Q1). Make a static prediction for every single row where that most frequent answer is your 1st guess, followed by the second most frequent, and then the third most frequent. What is the overall MAP@3 score of this "Majority Class" baseline on train.csv?

In [51]:
freq=train['answer'].value_counts()
top3=freq.index[:3].tolist()
print(f"Top 3 answers used for prediction: {top3}")
 
scores = []
for _,row in train.iterrows():
    score = ap_at_3(row['answer'], top3)
    scores.append(score)
 
map3 = round(np.mean(scores), 4)
print(f"Majority class baseline MAP@3: {map3}")

Top 3 answers used for prediction: ['B', 'C', 'A']
Majority class baseline MAP@3: 0.4212


10. The TF-IDF Pipeline: Build a basic pipeline that evaluates every row in train.csv. For each row, calculate the TF-IDF cosine similarity between the prompt and each of the 5 options. Sort these options from highest similarity to lowest to form your top 3 predictions. What is the final average MAP@3 score of this TF-IDF pipeline across the entire training set?  

In [52]:
scores = []
 
for _,row in train.iterrows():
    prompt_vec=vectorizer.transform([row['prompt']])
 
    sims = {}
    for opt in ['A', 'B', 'C', 'D', 'E']:
        opt_vec = vectorizer.transform([row[opt]])
        sims[opt] = cosine_similarity(prompt_vec, opt_vec)[0][0]
 
    top3=sorted(sims, key=sims.get, reverse=True)[:3]
    scores.append(ap_at_3(row['answer'], top3))
 
map3 = round(np.mean(scores),4)
print(f"TF-IDF pipeline MAP@3: {map3}")

TF-IDF pipeline MAP@3: 0.3119
